# Tutorial 3: Running Models on HPC4

**IEDA 4000I - Large Language Models and Operations Research**

This notebook is the class guide for Lab 3.

Important execution rule:
- Use this notebook for instructions and light checks. Keep your termianl window open on the side of the notebook for all GPU-critical steps.
- Run all GPU-critical steps in terminal (interactive SLURM session or sbatch).
- Do not rely on VSCode notebook kernel to inherit SLURM GPU allocation.


## 60-Minute Flow

| Time | Activity |
|---|---|
| 00:00-00:03 | Goals and success criteria |
| 00:03-00:20 | Section 1 SSH-first Git activity: Canvas folder -> local SSH clone -> push -> HPC4 SSH setup -> HPC4 clone/pull |
| 00:20-00:30 | Section 2 Lab 2 fast backfill (env + SLURM essentials) |
| 00:30-00:36 | Section 3 Hugging Face account/token (optional custom-model path) |
| 00:36-00:46 | Section 4-5 shared model path and interactive inference |
| 00:46-00:55 | Section 6 sbatch template and log verification |
| 00:55-00:58 | Section 7 Bridge to Lab 4 (LoRA/QLoRA with PEFT) |
| 00:58-01:00 | Section 8 Exit ticket submission |


# Section 0: What You Will Finish Today

By the end of this lab, you should complete these core items:

1. You have finished SSH-based Git sync workflow: local repo push to GitHub private repo, then HPC4 clone/pull via SSH.
2. You can run one GPU inference in terminal with `run_infer.py` using `/project/ugiedahpc4/ieda4000i/models/Qwen3-4B`.
3. You can submit one batch job with `run_infer.sbatch` and read the output log.
4. (Optional custom-model path) Hugging Face login works (`huggingface-cli whoami`).

Exit ticket evidence is listed in Section 8.


# Section 1: Git/GitHub Activity (SSH-First Workflow)

This is a required in-class activity. We will use SSH as the default path (not HTTPS), because repeated HTTPS login is inconvenient and credential cache may fail.

## Prelab checklist (before class)

- Make sure your **local machine** already has an SSH key.
- Make sure your local **public key** is already added in GitHub `Settings -> SSH and GPG keys`.
- If you are not sure how to do this, review Lab 2 related guidance (GitHub connection + SSH key sections).

## Step 1: Create a Private Repo on GitHub

1. Go to GitHub and create a **Private** repository.
2. Do not initialize with large templates; keep it clean for course materials.

## Step 2: SSH clone to your personal computer

On local machine:

```bash
git clone git@github.com:YOUR_USERNAME/YOUR_PRIVATE_REPO.git
cd YOUR_PRIVATE_REPO
```

Verify remote protocol is SSH:

```bash
git remote -v
# should show git@github.com:...
```

## Step 3: Copy `Lab 3` folder, then add / commit / push

```bash
# run in local repo root
# copy Canvas-distributed folder into Labs/Lab 3 first

git add "Labs/Lab 3"
git status
git commit -m "Add Lab 3 materials"
git push -u origin main
```

## Step 4: SSH to HPC4, generate SSH key, and add key to identity

On HPC4:

```bash
ssh YOUR_ITSC_USERNAME@hpc4.ust.hk
ssh-keygen -t ed25519 -C "YOUR_GITHUB_EMAIL"

# start ssh agent and add identity
eval "$(ssh-agent -s)"
ssh-add ~/.ssh/id_ed25519
```

Show your HPC4 public key:

```bash
cat ~/.ssh/id_ed25519.pub
```

## Step 5: Add HPC4 public key to GitHub Settings

1. Copy the full output from `~/.ssh/id_ed25519.pub` on HPC4.
2. Go to GitHub `Settings -> SSH and GPG keys`.
3. Click **New SSH key** and paste it.

## Step 6: On HPC4, clone/pull your GitHub repo via SSH

First-time clone on HPC4:

```bash
cd ~
git clone git@github.com:YOUR_USERNAME/YOUR_PRIVATE_REPO.git
cd YOUR_PRIVATE_REPO
ls "Labs/Lab 3"
```

If already cloned:

```bash
cd ~/YOUR_PRIVATE_REPO
git pull
ls "Labs/Lab 3"
```


## 1.7 If SSH to GitHub is blocked on HPC4: use SSH over port 443

Do this on HPC4 in `~/.ssh/config`:

```sshconfig
Host github.com
    Hostname ssh.github.com
    Port 443
    User git
```

Then test:

```bash
ssh -T git@github.com
```

If test passes, continue with SSH clone/pull.

Class policy for this activity:
- Prefer SSH workflow.
- Do not switch to HTTPS unless TA explicitly asks for a temporary workaround.


# Section 2: Lab 2 Fast Backfill

This section patches the parts from Lab 2 that are required for today's runtime flow:
- environment activation and package verification
- interactive and batch SLURM usage
- verification habit on HPC4 terminal

## 2.1 Environment activation and package checks
Run on HPC4 login node:

```bash
source "/opt/shared/.spack-edge/dist/bin/setup-env.sh" -y
conda activate ieda4000i # or your custom env name
python --version
python -c "import torch; print('torch:', torch.__version__)"
python -c "import transformers; print('transformers:', transformers.__version__)"
python -c "import huggingface_hub; print('huggingface_hub:', huggingface_hub.__version__)"
python -c "import accelerate; print('accelerate:', accelerate.__version__)"
python -c "import sentencepiece; print('sentencepiece:', sentencepiece.__version__)"
```


## 2.2 SLURM Backfill

This part is intentionally detailed because SLURM is the operational foundation for Lab 3 and Lab 4.

### 2.2.1 What is SLURM and why we must use it

**SLURM** (Simple Linux Utility for Resource Management) is the scheduler for shared HPC4 resources.

Key rule for this course:
- Never run heavy compute on login nodes.
- Always request compute resources through SLURM for GPU tasks.

### 2.2.2 Check available resources

Use `sinfo` to inspect partitions and current availability:

```bash
sinfo
```

Typical output looks like:

```text
PARTITION    AVAIL  TIMELIMIT  NODES  STATE NODELIST
intel           up   infinite      6   idle cpu[05-10]
amd             up   infinite     32   idle cpu[20,24-44,...]
gpu-a30         up   infinite      5   idle gpu[10-14]
gpu-l20         up   infinite      2   idle gpu[20-21]
gpu-rtx4090d    up   infinite      1   idle gpu41
gpu-rtx5880     up   infinite      2   idle gpu[39-40]
```

Node state quick reference:

| State | Meaning |
|---|---|
| `idle` | Available, no jobs running |
| `mix` | Partially used (some resources still available) |
| `alloc` | Fully allocated |
| `drain` | Unavailable / maintenance |

### 2.2.3 Start an interactive GPU session (development mode)

```bash
srun --partition=gpu-l20 --gres=gpu:1 --account=ugiedahpc4 --time=00:30:00 --pty bash
```

Flag explanation:

| Flag | Meaning |
|---|---|
| `--partition=gpu-l20` | Request L20 GPU partition |
| `--gres=gpu:1` | Request 1 GPU |
| `--account=ugiedahpc4` | Use course account |
| `--time=00:30:00` | Request 30-minute slot |
| `--pty bash` | Open interactive shell on allocated node |

Notes:
- Switch partition (for example `gpu-a30`, `gpu-rtx5880`) when `gpu-l20` is busy.
- Interactive sessions usually have an upper bound (commonly 4 hours on HPC4 policy).

Prompt change confirms allocation:

```text
[USERNAME@login1 ~]$  ->  [USERNAME@gpu-xx ~]$
```

### 2.2.4 After entering the GPU node

Re-activate conda environment:

```bash
conda activate ieda4000i
```

(Optional) If you find conda is not available, source Spack environment first:

```bash
source "/opt/shared/.spack-edge/dist/bin/setup-env.sh" -y
conda activate ieda4000i
```

Verify GPU access:

```bash
nvidia-smi
python -c "import torch; print('CUDA available:', torch.cuda.is_available()); print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A')"
```

Expected result:
- `nvidia-smi` shows allocated GPU.
- PyTorch prints `CUDA available: True` on compute node.

### 2.2.5 Exit interactive session

```bash
exit
```

This releases GPU resources and returns you to login node.

### 2.2.6 Batch jobs with `run_infer.py` (Lab 3 integrated)
You should use `sinfo` to check GPU availability before submitting batch jobs. If your preferred partition is busy, switch to another one.

For repeatable or longer runs, use `sbatch` with the provided script:
- Script: `Labs/Lab 3/scripts/run_infer.sbatch`
- Python runner: `Labs/Lab 3/scripts/run_infer.py`

Step 1: set runtime parameters with environment variables

```bash
export MODEL_PATH="/project/ugiedahpc4/ieda4000i/models/Qwen3-4B"
export PROMPT="Explain LP vs MILP in three concise bullet points."
export MAX_NEW_TOKENS=128
```

Step 2: submit job

```bash
sbatch "Labs/Lab 3/scripts/run_infer.sbatch"
```

Step 3: monitor queue

```bash
squeue -u $USER
```

Step 4: inspect output logs after completion

```bash
# Replace JOB_ID with actual id
cat lab3_infer_JOB_ID.out
cat lab3_infer_JOB_ID.err
```

What the batch script does internally:
1. Activates Spack + `ieda4000i` environment.
2. Prints job metadata and node info.
3. Runs:

```bash
python "Labs/Lab 3/scripts/run_infer.py" \
  --model_path "${MODEL_PATH}" \
  --prompt "${PROMPT}" \
  --max_new_tokens "${MAX_NEW_TOKENS}" \
  --temperature "${TEMPERATURE:-0.7}" \
  --top_p "${TOP_P:-0.9}" \
  --device "${DEVICE:-cuda}"
```

Step 5: cancel job if needed

```bash
scancel JOB_ID
```


# Section 3: Hugging Face Account and Token Setup (Optional)

Important: for Lab 3 main flow, we use pre-downloaded shared models on HPC4, so token is not required for the core demo.

Only use this section when you want custom models from Hugging Face Hub.

## 3.1 Create account and token (optional)

1. Register or sign in at https://huggingface.co
2. Create a **Read** token in account settings
3. Keep token private and never commit token text into Git

## 3.2 Login from HPC4 (only for custom model download)

```bash
source "/opt/shared/.spack-edge/dist/bin/setup-env.sh" -y
conda activate ieda4000i
pip install -U huggingface_hub

huggingface-cli login
huggingface-cli whoami
```

Optional cache location setup (if your home quota is tight):

```bash
export HF_HOME="/path/to/your/cache"
```


In [ ]:
# CPU-safe check cell: verifies required Python packages only.
import importlib

packages = ["torch", "transformers", "huggingface_hub", "accelerate", "sentencepiece"]
for pkg in packages:
    try:
        mod = importlib.import_module(pkg)
        print(f"OK  {pkg}: {getattr(mod, '__version__', 'unknown')}")
    except Exception as exc:
        print(f"MISSING/ERROR  {pkg}: {exc}")


# Section 4: Model Retrieval Strategy (Shared Path First)

## 4.1 Shared model directory on HPC4 (primary path)

TA-prepared shared models are stored here:

```bash
ls /project/ugiedahpc4/ieda4000i/models/
```

Current list:
- `DeepSeek-R1-0528-Qwen3-8B`
- `Qwen3-1.7B`
- `Qwen3-4B`
- `Qwen3-4B-Instruct-2507`

Main model for Lab 3:
- `/project/ugiedahpc4/ieda4000i/models/Qwen3-4B`

Fallback model (if needed):
- `/project/ugiedahpc4/ieda4000i/models/Qwen3-1.7B`

## 4.2 Class runtime strategy

1. Use shared path model first (no extra download time in class).
2. If main model has temporary issue, switch to shared fallback model.
3. Only if students need custom models, then use Hugging Face login + download.

## 4.3 Custom model download (optional branch)

Example (CLI):

```bash
huggingface-cli download Qwen/Qwen3-4B --local-dir ~/hf_models/Qwen3-4B
```

Python alternative:

```python
from huggingface_hub import snapshot_download
snapshot_download(repo_id="Qwen/Qwen3-4B", local_dir="~/hf_models/Qwen3-4B")
```

Then run inference with the downloaded local directory path.


In [ ]:
# Planning helper cell: choose model path before terminal execution.
MAIN_MODEL = "/project/ugiedahpc4/ieda4000i/models/Qwen3-4B"
FALLBACK_MODEL = "/project/ugiedahpc4/ieda4000i/models/Qwen3-1.7B"

# Optional Hugging Face model id (only for custom download path)
OPTIONAL_HF_ID = "Qwen/Qwen3-4B"

print("Main model (shared path):", MAIN_MODEL)
print("Fallback model (shared path):", FALLBACK_MODEL)
print("Optional HF model id:", OPTIONAL_HF_ID)
print("Note: GPU inference is executed in terminal, not in notebook cells.")


# Section 5: Interactive Inference (Terminal First, Shared Model Path)

From your course repo root on HPC4:

```bash
cd ~/ieda4000i/Labs/Lab\ 3/
source "/opt/shared/.spack-edge/dist/bin/setup-env.sh" -y

# Verify shared models exist
ls /project/ugiedahpc4/ieda4000i/models/

srun --partition=gpu-l20 --gres=gpu:1 --account=ugiedahpc4 --time=00:30:00 --pty bash
source "/opt/shared/.spack-edge/dist/bin/setup-env.sh" -y
conda activate ieda4000i

python "/scripts/run_infer.py" \
  --model_path "/project/ugiedahpc4/ieda4000i/models/Qwen3-4B" \
  --prompt "Explain the difference between LP and MILP in 3 bullet points." \
  --max_new_tokens 128 \
  --temperature 0.7 \
  --device cuda
```

You can also try the instruct model

```bash
python "Labs/Lab 3/scripts/run_infer.py" \
  --model_path "/project/ugiedahpc4/ieda4000i/models/Qwen3-4B-Instruct-2507" \
  --prompt "Explain the difference between LP and MILP in 3 bullet points." \
  --max_new_tokens 128 \
  --temperature 0.7 \
  --device cuda
```

To view the difference between the base and instruct models, compare outputs from the two runs above with the same prompt. The instruct model should provide a more concise and instruction-following response.

If CUDA is not available, the script prints a direct `srun` fix command.


# Section 6: Batch Inference with sbatch (run_infer.py step-by-step)

This section is the execution recipe students should follow exactly.

## 6.1 Prepare parameters (shared path first)

```bash
cd ~/ieda4000i/Labs/Lab\ 3/

export COURSE_ENV=ieda4000i # (optional) if you have a different env name, edit accordingly and paste into terminal
export MODEL_PATH="/project/ugiedahpc4/ieda4000i/models/Qwen3-4B" # primary shared model path, you can switch to the instruct model or fallback model if needed
export PROMPT="Give 3 concise tips for reproducible LLM experiments." # you can change the prompt to test different inputs
export MAX_NEW_TOKENS=128
export TEMPERATURE=0.7
export TOP_P=0.9
export DEVICE=cuda
```

Optional shared fallback model:

```bash
export MODEL_PATH="/project/ugiedahpc4/ieda4000i/models/Qwen3-1.7B"
```

Optional custom Hugging Face model (requires `huggingface-cli login` first):

```bash
export MODEL_PATH="Qwen/Qwen3-4B"
```

## 6.2 Submit and monitor

```bash
sbatch "./scripts/run_infer.sbatch"
```
You should see your job in the queue. Wait until it completes (state changes from `R` to `CD`).

Makesure to note your `JOB_ID` for log retrieval, after submission, it will print something like:

```text
Submitted batch job JOB_ID
```

Also, you can monitor the job status with:

```
(ieda4000i) [zhuangdr@login3 scripts]$ squeue -u $USER
             JOBID PARTITION     NAME     USER ST       TIME  NODES NODELIST(REASON)
            574447   gpu-l20 lab3_inf zhuangdr  R       0:43      1 gpu16
```

## 6.3 Read logs and verify result

```bash
# Replace JOB_ID with your actual id
cat lab3_infer_JOB_ID.out
cat lab3_infer_JOB_ID.err
```

Expected evidence in `.out`:
- `[INFO] Job id:` line
- `[INFO] Model path:` line
- `=== Generated Text ===` block from `run_infer.py`

## 6.4 How `run_infer.sbatch` maps to `run_infer.py`

The sbatch script passes variables into the Python CLI interface:

```bash
python "${SCRIPT_DIR}/run_infer.py" \
  --model_path "${MODEL_PATH}" \
  --prompt "${PROMPT}" \
  --max_new_tokens "${MAX_NEW_TOKENS}" \
  --temperature "${TEMPERATURE}" \
  --top_p "${TOP_P}" \
  --device "${DEVICE}"
```

This keeps one consistent execution path for both interactive runs and batch runs.

## 6.5 Common failure handling

- Pending too long: switch to another allowed partition and resubmit.
- CUDA not available: check `DEVICE` and verify job actually landed on GPU partition.
- Local model path not found: run `ls /project/ugiedahpc4/ieda4000i/models/` and fix path.
- Auth error when loading custom Hub model: re-run `huggingface-cli login`, then submit again.


# Section 7: Bridge to Lab 4 (LoRA and QLoRA Preview)

Lab 4 focus:
- Base model: Qwen3-4B
- Frameworks: Hugging Face Transformers + PEFT
- Training style: LoRA / QLoRA

Conceptual stack:
1. `transformers` loads tokenizer and base model
2. `peft` injects trainable low-rank adapters
3. `bitsandbytes` (for QLoRA) enables lower-memory quantized training
4. Training is submitted by `sbatch`, not by long notebook sessions

Minimal preview snippet (concept only):

```python
from peft import LoraConfig, get_peft_model

config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
    task_type="CAUSAL_LM",
)

# model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen3-4B")
# model = get_peft_model(model, config)
```


# Section 8: Exit Ticket

Submit the following evidence before leaving:

1. Local `git remote -v` shows SSH remote (`git@github.com:...`).
2. Local commit hash that adds `Labs/Lab 3`, and push success proof.
3. On HPC4, SSH key generated and added to GitHub (show key registration proof).
4. On HPC4, successful `git clone`/`git pull` via SSH and `ls "Labs/Lab 3"` output.
5. One `run_infer.py` output using `/project/ugiedahpc4/ieda4000i/models/Qwen3-4B`.
6. One `sbatch` job id and key lines from `lab3_infer_<jobid>.out`.
7. (Optional, custom-model track) `huggingface-cli whoami` output.

If any item fails, include the exact error line and your attempted fix command.


## Troubleshooting Quick Notes

| Symptom | Likely cause | Quick fix |
|---|---|---|
| `torch.cuda.is_available()` is False | Running on login node | Request GPU with `srun`, then reactivate env |
| `No such file or directory` for model path | Wrong shared model path | Check `ls /project/ugiedahpc4/ieda4000i/models/` and use full path |
| `401/403` when loading model | Missing HF token (custom Hub model only) | Run `huggingface-cli login`, verify with `whoami` |
| Job pending too long | Partition busy | Check `sinfo`, try another allowed GPU partition |
| `ModuleNotFoundError` | Missing package in env | Activate `ieda4000i`, reinstall package |
| Git push asks for password repeatedly | PAT not used/cached | Use PAT and set credential helper |
